# ESRS E1-2 — Climate-Related Physical Risk Assessment with CLIMADA

**Identification of Climate-Related Risks and Scenario Analysis**  
*Draft Simplified ESRS (EFRAG Technical Advice V1.5, 30 Nov 2025)*

This notebook runs the full CLIMADA **probabilistic** risk pipeline starting from nothing
but **site geolocations (WGS84 lat/lon)**. Everything else — the hazard event sets — comes
from **open-access data** served by the CLIMADA Data API (ETH Zurich):

| Hazard | Open-access source | CLIMADA `haz_type` |
|---|---|---|
| Flooding (river) | GloFAS / ISIMIP river-flood depth maps | `RF` |
| Extreme precipitation | Derived from the river-flood model (precip → flood → damage) | `RF` |
| Storms (tropical cyclone) | Synthetic tracks from IBTrACS, scaled by CMIP6 | `TC` |

The risk equation is **Risk = Hazard × Exposure × Vulnerability**. The output is the
**Expected Annual Loss (EAL)** per site — the core E1-2 metric — computed over a full
probabilistic event set (return-period flood maps / thousands of synthetic storms).

**Run matrix:** 3 hazards × 3 scenarios (SSP1-1.9, SSP2-4.5, SSP3-7.0) × 3 time horizons
= **27 runs per site**.

> Built and tested on **climada 6.1**. Uses `ImpactCalc(...).impact()` (the `Impact.calc()`
> method used in older guides is removed in climada 6.x).

## 1 · Imports

In [1]:
import pandas as pd
import numpy as np

from climada.entity import Exposures, ImpactFuncSet, ImpactFunc, ImpfTropCyclone
from climada.hazard import Hazard
from climada.engine import ImpactCalc
from climada.util.api_client import Client

## 2 · Load the exposure (your only input: geolocations)

Each row is one site. The **only data you need to supply are `latitude` / `longitude`**.

Because asset replacement values are company-specific (and often the one thing you don't
have yet), `value` defaults to **1.0** for every site. With unit value, the resulting EAL is
the **expected annual loss as a fraction of asset value** (i.e. the mean annual damage
ratio, 0–1). When you obtain real book/replacement values, drop them into the `value`
column and the same EAL becomes a currency figure — no other change needed.

The enrichment columns (`headcount`, `business_area`, `criticality`) are **not** used in the
CLIMADA math; they enrich the E1-2 *sensitivity* narrative (AR 6(b)) and carry through to the
final disclosure table. `country_iso3` is required to fetch the per-country open-access
hazard tiles. `impf_RF` / `impf_TC` link each site to its vulnerability curve (defined below).

In [2]:
# --- The only mandatory inputs: site geolocations (WGS84 decimal degrees) ---
sites = pd.DataFrame({
    'site_name':     ['Manila WH', 'London HQ', 'Singapore DC', 'Paris Office'],
    'latitude':      [14.5995,     51.5074,     1.3521,         48.8566],
    'longitude':     [120.9842,   -0.1278,      103.8198,       2.3522],
    'country_iso3':  ['PHL',       'GBR',        'SGP',          'FRA'],

    # Asset value: default 1.0 -> EAL is expressed as a fraction of asset value.
    # Replace with book / replacement cost (same currency) to get monetary EAL.
    'value':         [1.0,         1.0,          1.0,            1.0],

    # --- Enrichment (post-processing only; not used in CLIMADA's math) ---
    'headcount':     [120,         450,          85,             200],
    'business_area': ['Logistics', 'Corporate HQ','Data Center',  'Sales'],
    'criticality':   ['High',      'Critical',   'Critical',     'Medium'],

    # --- Vulnerability links: which impact-function id applies per hazard type ---
    'impf_RF':       [1, 1, 1, 1],   # river flood / extreme precip  -> ImpactFunc id 1 (RF)
    'impf_TC':       [1, 1, 1, 1],   # tropical cyclone              -> ImpactFunc id 1 (TC)
})

exp = Exposures(sites)
exp.check()
exp.gdf.head()

,site_name,country_iso3,value,headcount,business_area,criticality,impf_RF,impf_TC,geometry
0,Manila WH,PHL,1.0,120,Logistics,High,1,1,POINT (120.9842 14.5995)
1,London HQ,GBR,1.0,450,Corporate HQ,Critical,1,1,POINT (-0.1278 51.5074)
2,Singapore DC,SGP,1.0,85,Data Center,Critical,1,1,POINT (103.8198 1.3521)
3,Paris Office,FRA,1.0,200,Sales,Medium,1,1,POINT (2.3522 48.8566)


## 3 · Define the 27-run matrix (scenarios × time horizons × hazards)

**Scenarios.** CLIMADA's open hazard API is indexed by RCP labels. We map each SSP to the
closest available RCP that exists for **both** hazards so every run resolves to real data:

| ESRS scenario | ~Warming | River flood | Tropical cyclone | Note |
|---|---|---|---|---|
| SSP1-1.9 | +1.5 °C | `rcp26` | `rcp26` | SSP1-2.6 used as conservative proxy (document the ~0.3 °C) |
| SSP2-4.5 | +2.7 °C | `rcp60` | `rcp60` | `rcp45` is absent from the flood dataset → `rcp60` for both |
| SSP3-7.0 | +3.6 °C | `rcp85` | `rcp85` | High-emission scenario (satisfies E1-2 §16(a)(i)) |

**Time horizons.** The two hazards expose time differently, so each carries its own mapping:
river flood uses a `year_range` window; tropical cyclone uses a single `ref_year` snapshot.

In [3]:
# SSP -> RCP (intersection that exists for BOTH hazards in the open API)
scenarios = {
    'SSP1-1.9': 'rcp26',   # Paris-aligned (SSP1-2.6 proxy)
    'SSP2-4.5': 'rcp60',   # rcp45 not in flood data -> rcp60 for both
    'SSP3-7.0': 'rcp85',   # high emissions
}

# Per-hazard configuration.
#   time_prop  : the API property that carries the time dimension
#   timeframes : ESRS horizon -> the value for that property
#   extra_props: extra filters needed to resolve a SINGLE dataset
hazard_config = {
    'Flooding': {
        'haz_type':  'river_flood',   'tag': 'RF',  'impf_col': 'impf_RF',
        'time_prop': 'year_range',
        'timeframes': {'Short (0-3yr)': '2010_2030',
                       'Medium (3-10yr)': '2030_2050',
                       'Long (10+yr)': '2050_2070'},
        'extra_props': {},
    },
    'Extreme Precip.': {   # derived from the river-flood model (precip -> flood -> damage)
        'haz_type':  'river_flood',   'tag': 'RF',  'impf_col': 'impf_RF',
        'time_prop': 'year_range',
        'timeframes': {'Short (0-3yr)': '2010_2030',
                       'Medium (3-10yr)': '2030_2050',
                       'Long (10+yr)': '2050_2070'},
        'extra_props': {},
    },
    'Storms': {
        'haz_type':  'tropical_cyclone',  'tag': 'TC',  'impf_col': 'impf_TC',
        'time_prop': 'ref_year',
        'timeframes': {'Short (0-3yr)': '2040',
                       'Medium (3-10yr)': '2060',
                       'Long (10+yr)': '2080'},
        # random_walk = CLIMADA synthetic tracks from IBTrACS (resolves one dataset/country)
        'extra_props': {'model_name': 'random_walk'},
    },
}

COUNTRIES = sites['country_iso3'].unique().tolist()
print('Hazards :', list(hazard_config))
print('Scenarios:', list(scenarios))
print('Countries:', COUNTRIES)
print('Total runs:', len(hazard_config) * len(scenarios) * 3)

Hazards : ['Flooding', 'Extreme Precip.', 'Storms']
Scenarios: ['SSP1-1.9', 'SSP2-4.5', 'SSP3-7.0']
Countries: ['PHL', 'GBR', 'SGP', 'FRA']
Total runs: 27


## 4 · Build the impact (vulnerability) functions

- **Tropical cyclone** — Emanuel (2011) calibration, the field standard, mapping max wind
  speed (m/s) to damage fraction. `haz_type='TC'`, `id=1`.
- **River flood / extreme precip** — a JRC-style depth-damage curve mapping flood depth (m)
  to damage fraction. `haz_type='RF'`, `id=1`.

> The hazard tag must match the impact function's `haz_type`. The open river-flood datasets
> are tagged **`RF`** (not `FL`), so both the curve and the exposure column use `RF`.

In [4]:
# Tropical cyclone (Emanuel 2011) — haz_type 'TC', id 1
impf_tc = ImpfTropCyclone.from_emanuel_usa()

# River flood / extreme precip — depth (m) -> damage fraction. haz_type 'RF', id 1
impf_rf = ImpactFunc(
    id=1,
    haz_type='RF',
    name='Flood depth-damage (generic)',
    intensity=np.array([0, 0.5, 1, 1.5, 2, 3, 5, 10]),  # flood depth, metres
    mdd=np.array([0, 0.15, 0.30, 0.45, 0.55, 0.70, 0.85, 1.0]),  # mean damage degree
    paa=np.ones(8),                                       # % assets affected
    intensity_unit='m',
)

impf_sets = {
    'RF': ImpactFuncSet([impf_rf]),
    'TC': ImpactFuncSet([impf_tc]),
}
print('Impact functions ready:', {k: v.get_ids() for k, v in impf_sets.items()})

Impact functions ready: {'RF': {'RF': [1]}, 'TC': {'TC': [1]}}


## 5 · Hazard loader (open-access, per-country, cached)

Hazards are downloaded per country from the CLIMADA Data API and merged with `Hazard.concat`,
so one merged event set covers all sites. Two robustness features:

- **Graceful gaps.** Not every country has every hazard (e.g. London/Paris have no *tropical*
  cyclone dataset — extratropical windstorms need the Petals `StormEurope` module). Missing
  country/scenario tiles are skipped; affected sites simply receive no contribution there.
- **Caching.** `Flooding` and `Extreme Precip.` resolve to the *same* river-flood data, so each
  unique hazard is fetched only once.

In [5]:
client = Client()
_hazard_cache = {}

def load_hazard(cfg, rcp_code, time_value):
    """Return a merged Hazard for all COUNTRIES, or None if no data exists."""
    key = (cfg['haz_type'], rcp_code, time_value)
    if key in _hazard_cache:
        return _hazard_cache[key]

    parts = []
    for iso3 in COUNTRIES:
        props = {
            'country_iso3alpha': iso3,
            'climate_scenario': rcp_code,
            cfg['time_prop']: time_value,
            **cfg['extra_props'],
        }
        try:
            parts.append(client.get_hazard(cfg['haz_type'], properties=props))
        except Exception as exc:
            print(f"    - no {cfg['haz_type']} for {iso3} ({rcp_code}, {time_value}): "
                  f"{type(exc).__name__}")

    if not parts:
        haz = None
    elif len(parts) == 1:
        haz = parts[0]
    else:
        haz = Hazard.concat(parts)

    _hazard_cache[key] = haz
    return haz

## 6 · Run the 27-combination engine

For each (hazard × scenario × horizon) we fetch the open hazard set and compute the impact
with `ImpactCalc`. `imp.eai_exp[i]` is the Expected Annual Loss for site *i* (aligned to the
exposure order). Failed combinations are recorded with `eal = NaN` so the disclosure table
keeps every site/hazard row and the gaps are explicit rather than silently dropped.

In [6]:
# Purge stale cache locks left by interrupted TC downloads.
# Run this once before the main loop; safe to re-run (no-ops if file not in DB).
from pathlib import Path

_tc_stale = [
    r'C:\Users\engrj\climada\data\hazard\tropical_cyclone'
    r'\tropical_cyclone_10synth_tracks_150arcsec_rcp26_PHL_2040'
    r'\v2.1\tropical_cyclone_10synth_tracks_150arcsec_rcp26_PHL_2040.hdf5',
    r'C:\Users\engrj\climada\data\hazard\tropical_cyclone'
    r'\tropical_cyclone_10synth_tracks_150arcsec_rcp26_PHL_2060'
    r'\v2.1\tropical_cyclone_10synth_tracks_150arcsec_rcp26_PHL_2060.hdf5',
]

for _p in _tc_stale:
    try:
        Client.purge_cache_db(Path(_p))
        print(f'Purged: {Path(_p).name}')
    except Exception as _e:
        print(f'Skip (not in cache DB): {Path(_p).name} — {_e}')


Purged: tropical_cyclone_10synth_tracks_150arcsec_rcp26_PHL_2040.hdf5
Skip (not in cache DB): tropical_cyclone_10synth_tracks_150arcsec_rcp26_PHL_2060.hdf5 — <Model: Download> instance matching query does not exist:
SQL: SELECT "t1"."id", "t1"."url", "t1"."path", "t1"."startdownload", "t1"."enddownload" FROM "download" AS "t1" WHERE ("t1"."path" = ?) LIMIT ? OFFSET ?
Params: ['C:\\Users\\engrj\\climada\\data\\hazard\\tropical_cyclone\\tropical_cyclone_10synth_tracks_150arcsec_rcp26_PHL_2060\\v2.1\\tropical_cyclone_10synth_tracks_150arcsec_rcp26_PHL_2060.hdf5', 1, 0]


In [7]:
all_results = []
n_total = len(hazard_config) * len(scenarios) * 3
n = 0

for haz_name, cfg in hazard_config.items():
    for ssp_name, rcp_code in scenarios.items():
        for tf_name, tf_value in cfg['timeframes'].items():
            n += 1
            print(f"[{n:2d}/{n_total}] {haz_name} | {ssp_name} | {tf_name}")
            eal_by_site = {}
            try:
                hazard = load_hazard(cfg, rcp_code, tf_value)
                if hazard is None:
                    print('    -> no data for any site; recording NaN')
                else:
                    imp = ImpactCalc(exp, impf_sets[cfg['tag']], hazard).impact(
                        save_mat=False, assign_centroids=True)
                    for i, name in enumerate(sites['site_name']):
                        eal_by_site[name] = float(imp.eai_exp[i])
            except Exception as exc:
                print(f'    -> run failed: {type(exc).__name__}: {exc}')

            for _, row in sites.iterrows():
                all_results.append({
                    'site_name':     row['site_name'],
                    'latitude':      row['latitude'],
                    'longitude':     row['longitude'],
                    'country_iso3':  row['country_iso3'],
                    'headcount':     row['headcount'],
                    'business_area': row['business_area'],
                    'criticality':   row['criticality'],
                    'hazard':        haz_name,
                    'scenario':      ssp_name,
                    'timeframe':     tf_name,
                    'eal':           eal_by_site.get(row['site_name'], np.nan),
                })

results_df = pd.DataFrame(all_results)
results_df.to_csv('e1_2_climada_results.csv', index=False)
print(f'\nDone. {len(results_df)} rows -> e1_2_climada_results.csv')
results_df.head(12)

[ 1/27] Flooding | SSP1-1.9 | Short (0-3yr)
[ 2/27] Flooding | SSP1-1.9 | Medium (3-10yr)
[ 3/27] Flooding | SSP1-1.9 | Long (10+yr)
[ 4/27] Flooding | SSP2-4.5 | Short (0-3yr)
[ 5/27] Flooding | SSP2-4.5 | Medium (3-10yr)
[ 6/27] Flooding | SSP2-4.5 | Long (10+yr)
[ 7/27] Flooding | SSP3-7.0 | Short (0-3yr)
[ 8/27] Flooding | SSP3-7.0 | Medium (3-10yr)
[ 9/27] Flooding | SSP3-7.0 | Long (10+yr)
[10/27] Extreme Precip. | SSP1-1.9 | Short (0-3yr)
[11/27] Extreme Precip. | SSP1-1.9 | Medium (3-10yr)
[12/27] Extreme Precip. | SSP1-1.9 | Long (10+yr)
[13/27] Extreme Precip. | SSP2-4.5 | Short (0-3yr)
[14/27] Extreme Precip. | SSP2-4.5 | Medium (3-10yr)
[15/27] Extreme Precip. | SSP2-4.5 | Long (10+yr)
[16/27] Extreme Precip. | SSP3-7.0 | Short (0-3yr)
[17/27] Extreme Precip. | SSP3-7.0 | Medium (3-10yr)
[18/27] Extreme Precip. | SSP3-7.0 | Long (10+yr)
[19/27] Storms | SSP1-1.9 | Short (0-3yr)
[20/27] Storms | SSP1-1.9 | Medium (3-10yr)
2026-06-27 21:18:19,376 - climada.util.api_client - W

,site_name,latitude,longitude,country_iso3,headcount,business_area,criticality,hazard,scenario,timeframe,eal
0,Manila WH,14.5995,120.9842,PHL,120,Logistics,High,Flooding,SSP1-1.9,Short (0-3yr),0.000051
1,London HQ,51.5074,-0.1278,GBR,450,Corporate HQ,Critical,Flooding,SSP1-1.9,Short (0-3yr),0.000000
2,Singapore DC,1.3521,103.8198,SGP,85,Data Center,Critical,Flooding,SSP1-1.9,Short (0-3yr),0.000000
3,Paris Office,48.8566,2.3522,FRA,200,Sales,Medium,Flooding,SSP1-1.9,Short (0-3yr),0.001250
4,Manila WH,14.5995,120.9842,PHL,120,Logistics,High,Flooding,SSP1-1.9,Medium (3-10yr),0.000515
5,London HQ,51.5074,-0.1278,GBR,450,Corporate HQ,Critical,Flooding,SSP1-1.9,Medium (3-10yr),0.000000
6,Singapore DC,1.3521,103.8198,SGP,85,Data Center,Critical,Flooding,SSP1-1.9,Medium (3-10yr),0.000000
7,Paris Office,48.8566,2.3522,FRA,200,Sales,Medium,Flooding,SSP1-1.9,Medium (3-10yr),0.000000
8,Manila WH,14.5995,120.9842,PHL,120,Logistics,High,Flooding,SSP1-1.9,Long (10+yr),0.000115
9,London HQ,51.5074,-0.1278,GBR,450,Corporate HQ,Critical,Flooding,SSP1-1.9,Long (10+yr),0.000000


## 7 · Generate the E1-2 disclosure table

`eal` is the **Expected Annual Loss as a fraction of asset value** (because `value = 1.0`).
The pivot is the audit-facing artefact: per site & hazard, EAL across every scenario × horizon.
A site-level summary highlights the most exposed locations, and a scenario-level summary shows
how aggregate risk grows along each pathway.

In [8]:
if results_df['eal'].notna().any():
    pivot = results_df.pivot_table(
        index=['site_name', 'business_area', 'criticality', 'headcount', 'hazard'],
        columns=['scenario', 'timeframe'],
        values='eal',
        aggfunc='sum',
        dropna=False,
    )

    site_summary = (results_df.groupby('site_name')
        .agg(total_eal=('eal', 'sum'),
             peak_hazard_eal=('eal', 'max'),
             headcount=('headcount', 'first'),
             criticality=('criticality', 'first'))
        .sort_values('total_eal', ascending=False))

    scenario_summary = (results_df.groupby(['scenario', 'timeframe'])
        .agg(total_eal=('eal', 'sum'),
             max_site_eal=('eal', 'max'),
             sites_at_risk=('eal', lambda s: int((s > 0).sum())))
        .reset_index())

    with pd.ExcelWriter('e1_2_disclosure_table.xlsx') as xl:
        pivot.to_excel(xl, sheet_name='EAL_by_site_hazard')
        site_summary.to_excel(xl, sheet_name='Site_summary')
        scenario_summary.to_excel(xl, sheet_name='Scenario_summary')

    print('Wrote e1_2_disclosure_table.xlsx (3 sheets)\n')
    print('=== EAL (fraction of asset value) by site & hazard ===')
    display(pivot.round(4))
    print('\n=== Site summary (most exposed first) ===')
    display(site_summary.round(4))
    print('\n=== Scenario summary ===')
    print(scenario_summary.round(4).to_string(index=False))
else:
    print('No EAL values were produced — check network access to the CLIMADA Data API and '
          'that the requested scenario/year combinations exist for your countries.')

Wrote e1_2_disclosure_table.xlsx (3 sheets)

=== EAL (fraction of asset value) by site & hazard ===


scenario                                                             SSP1-1.9  \
timeframe                                                        Long (10+yr)   
site_name    business_area criticality headcount hazard                         
London HQ    Corporate HQ  Critical    85        Extreme Precip.          NaN   
                                                 Flooding                 NaN   
                                                 Storms                   NaN   
                                       120       Extreme Precip.          NaN   
                                                 Flooding                 NaN   
...                                                                       ...   
Singapore DC Sales         Medium      200       Flooding                 NaN   
                                                 Storms                   NaN   
                                       450       Extreme Precip.          NaN   
                                                 Flooding                 NaN   
                                                 Storms                   NaN   

scenario                                                                          \
timeframe                                                        Medium (3-10yr)   
site_name    business_area criticality headcount hazard                            
London HQ    Corporate HQ  Critical    85        Extreme Precip.             NaN   
                                                 Flooding                    NaN   
                                                 Storms                      NaN   
                                       120       Extreme Precip.             NaN   
                                                 Flooding                    NaN   
...                                                                          ...   
Singapore DC Sales         Medium      200       Flooding                    NaN   
                                                 Storms                      NaN   
                                       450       Extreme Precip.             NaN   
                                                 Flooding                    NaN   
                                                 Storms                      NaN   

scenario                                                                        \
timeframe                                                        Short (0-3yr)   
site_name    business_area criticality headcount hazard                          
London HQ    Corporate HQ  Critical    85        Extreme Precip.           NaN   
                                                 Flooding                  NaN   
                                                 Storms                    NaN   
                                       120       Extreme Precip.           NaN   
                                                 Flooding                  NaN   
...                                                                        ...   
Singapore DC Sales         Medium      200       Flooding                  NaN   
                                                 Storms                    NaN   
                                       450       Extreme Precip.           NaN   
                                                 Flooding                  NaN   
                                                 Storms                    NaN   

scenario                                                             SSP2-4.5  \
timeframe                                                        Long (10+yr)   
site_name    business_area criticality headcount hazard                         
London HQ    Corporate HQ  Critical    85        Extreme Precip.          NaN   
                                                 Flooding                 NaN   
                                                 Storms                   NaN   
                                       120       Extr


=== Site summary (most exposed first) ===


,total_eal,peak_hazard_eal,headcount,criticality
site_name,,,,
Manila WH,0.0825,0.0123,120,High
Paris Office,0.0200,0.0025,200,Medium
London HQ,0.0004,0.0001,450,Critical
Singapore DC,0.0000,0.0000,85,Critical



=== Scenario summary ===
scenario       timeframe  total_eal  max_site_eal  sites_at_risk
SSP1-1.9    Long (10+yr)     0.0023        0.0010              5
SSP1-1.9 Medium (3-10yr)     0.0109        0.0098              4
SSP1-1.9   Short (0-3yr)     0.0125        0.0099              6
SSP2-4.5    Long (10+yr)     0.0143        0.0119              6
SSP2-4.5 Medium (3-10yr)     0.0148        0.0108              6
SSP2-4.5   Short (0-3yr)     0.0134        0.0100              6
SSP3-7.0    Long (10+yr)     0.0066        0.0025              4
SSP3-7.0 Medium (3-10yr)     0.0160        0.0123              6
SSP3-7.0   Short (0-3yr)     0.0122        0.0109              6


## 8 · ESRS E1-2 datapoint mapping

How the outputs above satisfy the draft E1-2 requirements:

| Paragraph | Requirement | Satisfied by |
|---|---|---|
| §14 | Classify each risk as physical / transition | `hazard` column — all three are **physical** risks |
| §15(a) | Methodology for exposure to hazards | CLIMADA (ETH Zurich, peer-reviewed, used by EIOPA) + this notebook |
| §15(b) | Transition events & trends | **Out of scope** — CLIMADA is physical-risk only; needs a separate assessment |
| §16(a)(i) | ≥1 high-emission scenario | **SSP3-7.0** (`rcp85`) |
| §16(a)(iii) | Temperature projections | SSP→°C table in §3 |
| §16(b) | Scope of operations | `site_name`, `latitude`, `longitude`, `country_iso3` |
| §16(c) | Key assumptions | Impact functions (§4), unit asset value, RCP proxies (§3) |
| §16(d) | Time period of analysis | `timeframe` column / Short–Medium–Long horizons |
| AR 6(a) | Screen which assets are exposed | `eal > 0` flags exposed sites |
| AR 6(b) | Sensitivity considering location | `eal` + `headcount`, `business_area`, `criticality` |

**Documented limitations** (state these in the narrative for audit defensibility):

1. *Extreme precipitation* is derived from the river-flood model (precip → flood → damage),
   per the standard CLIMADA approach — so its EAL mirrors the flood result.
2. *Tropical cyclone* covers only cyclone-prone basins; European sites (London, Paris) show no
   TC risk here. Extratropical **windstorm** risk for Europe needs the Petals `StormEurope`
   module — a separate run.
3. SSP2-4.5 uses `rcp60` (flood data lacks `rcp45`) and SSP1-1.9 uses the SSP1-2.6 proxy;
   both substitutions are conservative and should be disclosed.
4. With `value = 1.0`, EAL is a damage **fraction**; multiply by real asset values for monetary
   loss before reporting figures externally.